# Insurance Benefits Prediction & Data Obfuscation

**Goal:** Apply machine learning and linear algebra to solve four business tasks for an insurance company — from finding similar customers to protecting personal data without degrading model performance.

**Dataset:** 5,000 insurance customers with features: gender, age, income, family members, and insurance benefits received.

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import math
import seaborn as sns
import matplotlib.pyplot as plt

import sklearn.linear_model
import sklearn.metrics
import sklearn.preprocessing

from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('/datasets/insurance_us.csv')
df = df.rename(columns={
    'Gender': 'gender',
    'Age': 'age',
    'Salary': 'income',
    'Family members': 'family_members',
    'Insurance benefits': 'insurance_benefits'
})
df['age'] = df['age'].astype(int)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

**Data quality check:** No missing values or duplicate records. All features are within plausible ranges (age: 18–65, income: 5,300–79,000, family members: 0–6).

## 2. Exploratory Data Analysis

In [ ]:
g = sns.pairplot(df, kind='hist')
g.fig.set_size_inches(12, 12)
plt.suptitle('Pairplot — Feature Distributions and Relationships', y=1.01)
plt.show()

## Task 1 — Finding Similar Customers (kNN)

In [ ]:
feature_names = ['gender', 'age', 'income', 'family_members']

def get_knn(df, n, k, metric):
    nbrs = NearestNeighbors(n_neighbors=k, metric=metric).fit(df[feature_names])
    nbrs_distances, nbrs_indices = nbrs.kneighbors([df.iloc[n][feature_names]], k, return_distance=True)
    df_res = pd.concat([
        df.iloc[nbrs_indices[0]],
        pd.DataFrame(nbrs_distances.T, index=nbrs_indices[0], columns=['distance'])
    ], axis=1)
    return df_res

In [ ]:
# Scale data with MaxAbsScaler
scaler = MaxAbsScaler().fit(df[feature_names])
df_scaled = df.copy()
df_scaled[feature_names] = scaler.transform(df[feature_names])

In [ ]:
print('Unscaled — Euclidean:')
display(get_knn(df, n=0, k=5, metric='euclidean'))

print('Unscaled — Manhattan:')
display(get_knn(df, n=0, k=5, metric='manhattan'))

In [ ]:
print('Scaled — Euclidean:')
display(get_knn(df_scaled, n=0, k=5, metric='euclidean'))

print('Scaled — Manhattan:')
display(get_knn(df_scaled, n=0, k=5, metric='manhattan'))

**Findings:**
- **Unscaled data distorts kNN results** — `income` (range: 5k–79k) dominates the distance calculation, overshadowing lower-magnitude features like age or family members.
- **After scaling**, neighbors are meaningfully similar across all features.
- Euclidean and Manhattan metrics return similar neighbor sets after scaling, with minor ordering differences.

## Task 2 — Insurance Benefit Classification (kNN vs Dummy)

In [ ]:
df['insurance_benefits_received'] = (df['insurance_benefits'] > 0).astype(int)

X = df[feature_names]
y = df['insurance_benefits_received']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12345)

scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def eval_classifier(y_true, y_pred, label=''):
    f1 = sklearn.metrics.f1_score(y_true, y_pred)
    print(f'{label} F1: {f1:.2f}')

In [ ]:
results = {'k': [], 'F1 (unscaled)': [], 'F1 (scaled)': []}

for k in range(1, 11):
    # Unscaled
    m1 = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    f1_raw = sklearn.metrics.f1_score(y_test, m1.predict(X_test))

    # Scaled
    m2 = KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train)
    f1_scaled = sklearn.metrics.f1_score(y_test, m2.predict(X_test_scaled))

    results['k'].append(k)
    results['F1 (unscaled)'].append(round(f1_raw, 2))
    results['F1 (scaled)'].append(round(f1_scaled, 2))

pd.DataFrame(results).set_index('k')

In [ ]:
# Dummy model benchmark
def rnd_model_predict(P, size, seed=42):
    rng = np.random.default_rng(seed=seed)
    return rng.binomial(n=1, p=P, size=size)

base_rate = df['insurance_benefits_received'].mean()
print('Dummy model F1 scores:')
for P in [0, base_rate, 0.5, 1]:
    y_pred_rnd = rnd_model_predict(P, size=len(df))
    f1 = sklearn.metrics.f1_score(df['insurance_benefits_received'], y_pred_rnd)
    print(f'  P={P:.2f} → F1: {f1:.2f}')

**Findings:**
- **Scaled kNN (k=1) achieved F1 = 0.97**, demonstrating strong classification performance.
- **Unscaled kNN** peaks at F1 = 0.61 (k=1) and degrades quickly with larger k.
- **Dummy model** maxes out at F1 ≈ 0.20, confirming that kNN significantly outperforms random prediction.

## Task 3 — Insurance Benefits Regression (Custom Linear Regression)

In [ ]:
class MyLinearRegression:
    """Linear Regression implemented via the Normal Equation: w = (XᵀX)⁻¹ Xᵀy"""

    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        X2 = np.append(np.ones([len(X), 1]), X, axis=1)
        self.weights = np.linalg.inv(X2.T @ X2) @ X2.T @ y

    def predict(self, X):
        X2 = np.append(np.ones([len(X), 1]), X, axis=1)
        return X2 @ self.weights


def eval_regressor(y_true, y_pred, label=''):
    rmse = math.sqrt(sklearn.metrics.mean_squared_error(y_true, y_pred))
    r2 = sklearn.metrics.r2_score(y_true, y_pred)
    print(f'{label} RMSE: {rmse:.2f} | R²: {r2:.2f}')

In [ ]:
X = df[feature_names].to_numpy()
y = df['insurance_benefits'].to_numpy()

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12345)

# Original data
lr = MyLinearRegression()
lr.fit(x_train, y_train)
eval_regressor(y_test, lr.predict(x_test), label='Original:')

# Scaled data
scaler = MaxAbsScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

lr_scaled = MyLinearRegression()
lr_scaled.fit(x_train_scaled, y_train)
eval_regressor(y_test, lr_scaled.predict(x_test_scaled), label='Scaled: ')

**Finding:** RMSE and R² are identical with and without scaling — linear regression is scale-invariant, as scaling only transforms the weight magnitudes, not the predictions.

## Task 4 — Data Obfuscation via Matrix Transformation

### Analytical Proof

Given the transformation `X_obf = X @ P` where P is an invertible matrix, the new weights become:

```
w_obf = ((XP)ᵀ(XP))⁻¹ (XP)ᵀ y
      = (PᵀXᵀXP)⁻¹ Pᵀ Xᵀ y
      = P⁻¹ (XᵀX)⁻¹ (Pᵀ)⁻¹ Pᵀ Xᵀ y
      = P⁻¹ w
```

Predictions with obfuscated data:
```
ŷ_obf = X_obf @ w_obf = XP @ P⁻¹w = Xw = ŷ
```

**Conclusion:** The transformation cancels out exactly. Predictions and metrics remain unchanged.

In [ ]:
# Generate random invertible matrix P
rng = np.random.default_rng(seed=42)
while True:
    P = rng.random((X.shape[1], X.shape[1]))
    if np.linalg.det(P) != 0:
        break

X_obf = X @ P

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12345)
x_train_obf, x_test_obf, _, _ = train_test_split(X_obf, y, test_size=0.3, random_state=12345)

# Original
lr = MyLinearRegression()
lr.fit(x_train, y_train)
eval_regressor(y_test, lr.predict(x_test), label='Original:  ')

# Obfuscated
lr_obf = MyLinearRegression()
lr_obf.fit(x_train_obf, y_train)
eval_regressor(y_test, lr_obf.predict(x_test_obf), label='Obfuscated:')

**Verification:** RMSE and R² are identical before and after obfuscation, confirming the analytical proof.

## Conclusion

| Task | Method | Key Result |
|---|---|---|
| Similar Customers | kNN | Scaling is required — income dominates unscaled distances |
| Benefit Classification | kNN (scaled, k=1) | F1 = 0.97 vs. dummy max of 0.20 |
| Benefit Regression | Custom Linear Regression | RMSE = 0.34, scale-invariant |
| Data Obfuscation | Matrix transformation (X @ P) | RMSE unchanged — privacy preserved without quality loss |

This project demonstrates that **data privacy and model performance are not mutually exclusive**. Matrix obfuscation effectively masks personal information while leaving linear regression predictions completely intact — both analytically and empirically.